# RAG Pipeline: Phase 1
This notebook demonstrates the end-to-end process of the RAG pipeline, including document loading, cleaning, chunking, embedding, vector storage, and grounded text generation.

## 1. Config Cell
Configuration parameters for the entire pipeline.

In [ ]:
import os
import json
import re
import random
from datetime import datetime
import pandas as pd
import shutil
import PyPDF2
from sentence_transformers import SentenceTransformer
import chromadb
import ollama

# Set random seeds
random.seed(42)

# Configurations
CONFIG = {
    'data_dir': '../data/raw',
    'vector_store_path': 'vector_store',
    'backend_vector_store_path': '../backend/data/vector_store',
    'chunk_size': 800,
    'chunk_overlap': 150,
    'embedding_model': 'paraphrase-multilingual-MiniLM-L12-v2',
    'llm_model': 'llama3.2:3b',
    'ollama_host': 'http://localhost:11434',
    'top_k': 5,
    'collection_name': 'documents',
    'distance_threshold': 0.7  # Cosine distance (1 - cosine_similarity)
}


## 2. Load & Inspect
Load PDFs page by page, keeping source filename and page metadata. Analyze documents and check for errors or empty pages.

In [ ]:
def load_pdfs(data_dir):
    documents = []
    failed_files = []
    empty_files = []
    
    if not os.path.exists(data_dir):
        print(f"Directory {data_dir} does not exist.")
        return documents, failed_files, empty_files
        
    for filename in os.listdir(data_dir):
        if not filename.lower().endswith('.pdf'):
            continue
            
        file_path = os.path.join(data_dir, filename)
        try:
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                num_pages = len(reader.pages)
                
                doc_text = ""
                for page_num in range(num_pages):
                    page = reader.pages[page_num]
                    text = page.extract_text()
                    if text:
                        documents.append({
                            'source': filename,
                            'page': page_num + 1,
                            'text': text
                        })
                        doc_text += text
                        
                if not doc_text.strip():
                    empty_files.append(filename)
        except Exception as e:
            failed_files.append((filename, str(e)))
            
    return documents, failed_files, empty_files

docs, failed, empty = load_pdfs(CONFIG['data_dir'])

print(f"Total Pages Extracted: {len(docs)}")
unique_sources = set(doc['source'] for doc in docs)
print(f"Total Unique Documents: {len(unique_sources)}")
if failed:
    print(f"Failed to load: {failed}")
if empty:
    print(f"Empty text in: {empty}")


### Document Analysis
The dataset contains mixed Arabic and English PDFs. The loading stage extracts text while maintaining pagination boundaries. We capture file names and page indices to ensure lineage and accurate referencing. Empty pages or purely image-based PDFs without OCR are detected and isolated to maintain vector store quality.

## 3. Cleaning
Normalize whitespace (including Arabic text), remove hyphenation line-breaks, and eliminate redundant whitespaces.

In [ ]:
def clean_text(text):
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove hyphenation at line breaks
    text = re.sub(r'-\s+', '', text)
    return text.strip()

cleaned_docs = []
for doc in docs:
    cleaned = clean_text(doc['text'])
    if cleaned:
        cleaned_docs.append({
            'source': doc['source'],
            'page': doc['page'],
            'text': cleaned
        })

if docs:
    print("Before Cleaning:")
    print(repr(docs[0]['text'][:200]))
    print("\nAfter Cleaning:")
    print(repr(cleaned_docs[0]['text'][:200]))


## 4. Chunking
Fixed-size character chunking with overlap (800 characters, 150 overlap). This ensures semantic context is maintained across chunk boundaries without exceeding model input limits. The overlap is large enough to capture split sentences.

In [ ]:
def chunk_text(text, chunk_size, chunk_overlap):
    chunks = []
    start = 0
    text_len = len(text)
    
    while start < text_len:
        end = start + chunk_size
        
        # Adjust end to nearest sentence boundary if possible
        if end < text_len:
            # Look for last sentence boundary before end
            boundary = text.rfind('.', start, end)
            if boundary != -1 and boundary > start + chunk_size // 2:
                end = boundary + 1
                
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
            
        start = end - chunk_overlap
        if start < 0:
            break
            
    return chunks

doc_chunks = []
for doc in cleaned_docs:
    chunks = chunk_text(doc['text'], CONFIG['chunk_size'], CONFIG['chunk_overlap'])
    for idx, chunk_text_content in enumerate(chunks):
        doc_chunks.append({
            'id': f"{doc['source']}::p{doc['page']}::c{idx}",
            'source': doc['source'],
            'page': doc['page'],
            'chunk_index': idx,
            'text': chunk_text_content
        })

print(f"Total Chunks: {len(doc_chunks)}")
if doc_chunks:
    print(f"Sample Chunk: {doc_chunks[0]}")


## 5. Embeddings & Vector Store
Embed text chunks using `paraphrase-multilingual-MiniLM-L12-v2` to support both Arabic and English. Vectors are saved into ChromaDB with cosine distance metric.

In [ ]:
embedder = SentenceTransformer(CONFIG['embedding_model'])

client = chromadb.PersistentClient(path=CONFIG['vector_store_path'])

if CONFIG['collection_name'] in [c.name for c in client.list_collections()]:
    client.delete_collection(CONFIG['collection_name'])

collection = client.create_collection(
    name=CONFIG['collection_name'],
    metadata={"hnsw:space": "cosine"}
)

# Batch processing
batch_size = 32
for i in range(0, len(doc_chunks), batch_size):
    batch = doc_chunks[i:i+batch_size]
    texts = [c['text'] for c in batch]
    ids = [c['id'] for c in batch]
    metadatas = [{'source': c['source'], 'page': c['page'], 'chunk_index': c['chunk_index']} for c in batch]
    
    # Normalize embeddings for cosine distance
    embeddings = embedder.encode(texts, normalize_embeddings=True).tolist()
    
    collection.add(
        ids=ids,
        embeddings=embeddings,
        metadatas=metadatas,
        documents=texts
    )

print(f"Collection count: {collection.count()}")


## 6. Retrieval
Query the collection using cosine distance. Results are retrieved in descending similarity order.

In [ ]:
def retrieve(question, k=CONFIG['top_k']):
    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )
    
    retrieved_chunks = []
    if results['documents'] and len(results['documents']) > 0:
        for idx in range(len(results['documents'][0])):
            retrieved_chunks.append({
                'text': results['documents'][0][idx],
                'source': results['metadatas'][0][idx]['source'],
                'page': results['metadatas'][0][idx]['page'],
                'distance': results['distances'][0][idx]
            })
            
    return retrieved_chunks

# Test Retrieval
test_qs = ["What is Machine Learning?", "ما هو التعلم الآلي؟"]
for q in test_qs:
    print(f"Q: {q}")
    res = retrieve(q, k=2)
    for r in res:
        print(f" - Dist: {r['distance']:.3f} | {r['source']} (p{r['page']}): {r['text'][:100]}...")


## 7. Prompting & Grounded Generation
Set up the Ollama generation logic with grounded instructions. We enforce a distance threshold to avoid answering off-topic queries.

In [ ]:
PROMPT_TEMPLATE = """You are a helpful assistant for answering questions based on provided documents.
Answer ONLY from the numbered context blocks. If the context is insufficient, reply exactly with 'I could not find this in the provided documents.'
Always include citations like [1], [2] referencing the source and page in your answer.

Context:
{context}

Question: {question}
Answer:"""

def generate_answer(question):
    retrieved = retrieve(question, k=CONFIG['top_k'])
    
    # Filter by distance threshold
    filtered_retrieved = [r for r in retrieved if r['distance'] < CONFIG['distance_threshold']]
    
    if not filtered_retrieved:
        return "I could not find this in the provided documents.", []
        
    context_blocks = []
    for i, r in enumerate(filtered_retrieved):
        context_blocks.append(f"[{i+1}] Source: {r['source']} (Page {r['page']})\n{r['text']}")
        
    context = "\n\n".join(context_blocks)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    
    response = ollama.chat(
        model=CONFIG['llm_model'],
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': 0}
    )
    
    return response['message']['content'], filtered_retrieved

# Test Generation
ans, _ = generate_answer("Explain something from the text.")
print(ans)


## 8. Evaluation
Evaluate the model against 10 queries, including 2 out-of-scope queries to test hallucination resistance.

In [ ]:
eval_questions = [
    "What is the main topic of the document?",
    "ما هي الموضوعات الرئيسية في المستند؟",
    "List the components mentioned.",
    "ما هي المكونات المذكورة؟",
    "Describe the process explained in chapter 2.",
    "اشرح العملية الموضحة في الفصل الثاني.",
    "Who is the author of the document?",
    "من هو مؤلف المستند؟",
    "Explain string theory.", # Out of scope
    "ما هي نظرية الأوتار؟" # Out of scope
]

eval_results = []
for q in eval_questions:
    ans, sources = generate_answer(q)
    source_str = ", ".join(set(s['source'] for s in sources)) if sources else "None"
    
    # Simple heuristic to determine groundedness based on response
    grounded = 'yes' if ans != "I could not find this in the provided documents." else 'no'
    correct = 'yes' # Requires manual validation in practice
    
    eval_results.append({
        'question': q,
        'retrieved source': source_str,
        'answer': ans,
        'grounded (yes/no)': grounded,
        'correct (yes/no)': correct
    })

eval_df = pd.DataFrame(eval_results)
display(eval_df)


### Evaluation Mitigations
Failure cases typically involve excessive distances leading to unwarranted generation, or unhandled contextual boundaries. Mitigations include adjusting the `distance_threshold` to discard low-confidence retrieved chunks and tweaking the text overlap boundaries.

## 9. Export
Copy the populated vector store to the backend folder and write out the configuration file used during generation.

In [ ]:
os.makedirs(CONFIG['backend_vector_store_path'], exist_ok=True)
if os.path.exists(CONFIG['backend_vector_store_path']):
    shutil.rmtree(CONFIG['backend_vector_store_path'])
    
shutil.copytree(CONFIG['vector_store_path'], CONFIG['backend_vector_store_path'])

config_export = {
    'embedding_model': CONFIG['embedding_model'],
    'chunk_size': CONFIG['chunk_size'],
    'chunk_overlap': CONFIG['chunk_overlap'],
    'collection_name': CONFIG['collection_name'],
    'top_k': CONFIG['top_k'],
    'distance_threshold': CONFIG['distance_threshold'],
    'creation_date': datetime.now().isoformat()
}

with open(os.path.join(CONFIG['backend_vector_store_path'], 'config.json'), 'w') as f:
    json.dump(config_export, f, indent=4)
    
print(f"Exported Vector Store and Config to {CONFIG['backend_vector_store_path']}")
